<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/04_phase3/MaxVit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – MaxVit Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions (text_qwen3-4b)
  - Hybrid captions (hybrid_gemma3-4b)
  - Vision-generated captions (vision_gemma3-4b)
  
The impact of different lightweight text encoders (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is MaxVit, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [1]:
!pip install -q timm transformers accelerate wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs("/content/drive/MyDrive/DI725_term_project_phase3_predictions", exist_ok=True)

In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines the main experimental settings used throughout the notebook, including sample size, image size, batch size, number of epochs, learning rate, weight decay, early stopping patience, and random seed.

The target variables are the seven classes: Tree, Shrub, Grass, Crop, Built-up, Barren, and Water. The text_scale parameter controls the strength of the textual contribution during gated fusion.

In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "maxvit_tiny_tf_224",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2764/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2764/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2764/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
df["hybrid_gemma3_4b_clean"].head(10)

,hybrid_gemma3_4b_clean
0,the image depicts a landscape dominated by ext...
1,the image depicts a largely arid landscape dom...
2,the image depicts a landscape dominated by ext...
3,the image depicts a valley dominated by dense ...
4,the image depicts a coastal area dominated by ...
5,the image depicts a landscape dominated by cul...
6,the image depicts a landscape dominated by a r...
7,the image depicts a hilly landscape dominated ...
8,the image depicts a landscape dominated by ext...
9,the image depicts a landscape dominated by ext...


In [11]:
df["hybrid_gemma3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_2764/1445695331.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hybrid_gemma3_4b_clean"].str.contains(


np.int64(0)

In [12]:
df["text_qwen3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_2764/895902091.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["text_qwen3_4b_clean"].str.contains(


np.int64(0)

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land-cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

## Train / Validation / Test Split

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [13]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

sample_df = sample_df.reset_index(drop=True)

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land cover classes remain consistent across splits, preventing bias in training or evaluation.

In [14]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [15]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [16]:
IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

This dataset class prepares inputs for multimodal training. For each sample the image is loaded and transformed and target values are extracted as a multi-output regression vector.
If a text column is provided, captions are tokenized using a pretrained tokenizer

The dataset returns, image tensor, target vector, tokenized text inputs. This structure enables joint processing of image and text features in the model.

Dataset Classes

In [17]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [18]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (MaxVit) is used to extract image features, which are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, fully connected layers,ReLU activation and dropout for regularization

This baseline allows comparison against multimodal models to measure the contribution of textual information. The backbone is initialized with pretrained weights and fine-tuned during training.

In [19]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (MaxVit) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT) for caption representation
- A projection layer to map text features into the image feature space

The text encoder is kept frozen during training to reduce computational cost and ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are concatenated and passed through a learnable gate. The gate controls how much textual information contributes to the final representation. Moreover, a scaling factor further adjusts the strength of text features

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

This design allows the model to adaptively use textual information depending on its relevance.

In [20]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [21]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename="RemoteCLIP-ViT-B-32.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [22]:
class MaxVitRemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Functions

This section defines evaluation procedures for both image only and multimodal models.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean Squared Error (RMSE) is also computed for additional analysis. Furthermore, Class-wise MAE is calculated to analyze performance across land-cover categories

These metrics provide both overall and class-level performance insights.

In [23]:
@torch.no_grad()
def evaluate_image_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

In [24]:
@torch.no_grad()
def evaluate_text_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        input_ids = batch["input_ids"].to(config["device"])
        attention_mask = batch["attention_mask"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

In [25]:
@torch.no_grad()
def evaluate_remoteclip_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        texts = batch["text"]
        targets = batch["target"].to(config["device"])

        preds = model(images, texts)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training: Image-Only Model

This function defines the training loop for the image-only baseline. The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with weight decay is used.

Training progress is logged using Weights & Biases (W&B).

In [26]:
def train_image_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_image_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

## Training: Multimodal Model

This function extends the training process to multimodal inputs.

Differences from image only training is that the image and tokenized text inputs are used. Moreover, the multimodal forward pass combines features using gated fusion.

The same training configuration is maintained. This ensures a fair comparison between image only and multimodal models.

In [27]:
def train_text_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            input_ids = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images, input_ids, attention_mask)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_text_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

In [28]:
def train_remoteclip_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            texts = batch["text"]
            targets = batch["target"].to(config["device"])

            preds = model(images, texts)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_remoteclip_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

In [29]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiment: Image Only Baseline

This function runs the image only experiment using the MaxVit backbone.

Metrics (MAE, RMSE, and class-wise MAE) are logged using Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [30]:
def run_maxvit_image_only(save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "maxvit_image_only"
    run_config["fusion"] = "none"
    run_config["text_column"] = "none"
    run_config["text_model"] = "none"

    wandb.init(
        project="DI725-Project-landcover-regression_phase3_experiments",
        name="maxvit_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_image_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_image_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_image_only.csv",
            index=False
        )

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": "maxvit_image_only",
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual information.

Different caption sources and text encoders (MiniLM, DistilBERT) are utilized

For each configuration the model is initialized with a gated fusion mechanism, text contribution is controlled via a scaling factor and the same training and evaluation pipeline is applied

Results are logged to W&B, enabling comparison across different text sources and encoders. This setup allows systematic evaluation of how textual information impacts model performance.

In [31]:
def run_maxvit_transformer_text(text_col, text_model, text_scale=0.7, wandb_project="DI725-Project-landcover-regression_phase3_experiments",save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "maxvit_transformer_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = text_model
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"maxvit_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(
        project=wandb_project,
        name=run_name,
        config=run_config,
        reinit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_text_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_text_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv",
            index=False
        )
    wandb.log({
        "test_loss":    test_loss,
        "test_mae":     test_mae,
        "test_rmse":    test_rmse,
        "best_val_mae": best_val_mae
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })

    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [32]:
def run_maxvit_remoteclip_text(text_col, text_scale=0.7, wandb_project="DI725-Project-landcover-regression_phase3_experiments", save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "maxvit_remoteclip_text"
    run_config["text_column"] = text_col
    run_config["text_model"]  = "RemoteCLIP ViT-B-32"
    run_config["text_scale"]  = text_scale
    run_config["fusion"]      = "gated_frozen_scaled"

    run_name = f"maxvit_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(
        project=wandb_project,
        name=run_name,
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = MaxVitRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_remoteclip_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_remoteclip_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv",
            index=False
        )
    wandb.log({
        "test_loss":    test_loss,
        "test_mae":     test_mae,
        "test_rmse":    test_rmse,
        "best_val_mae": best_val_mae
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })

    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [33]:
results = []

## Image Only Experiment

In [34]:
results.append(run_maxvit_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 11.6082 | Val MAE: 8.7338 | Val RMSE: 19.7980
Epoch 2/15 | Train Loss: 7.1176 | Val MAE: 4.6686 | Val RMSE: 11.1178
Epoch 3/15 | Train Loss: 4.8426 | Val MAE: 3.5230 | Val RMSE: 8.8596
Epoch 4/15 | Train Loss: 4.0431 | Val MAE: 3.2130 | Val RMSE: 8.4170
Epoch 5/15 | Train Loss: 3.7312 | Val MAE: 2.7968 | Val RMSE: 7.6882
Epoch 6/15 | Train Loss: 3.4214 | Val MAE: 2.9710 | Val RMSE: 7.8397
Epoch 7/15 | Train Loss: 3.1555 | Val MAE: 2.9828 | Val RMSE: 7.4997
Epoch 8/15 | Train Loss: 2.9557 | Val MAE: 2.6456 | Val RMSE: 7.3262
Epoch 9/15 | Train Loss: 2.7615 | Val MAE: 2.4525 | Val RMSE: 6.3472
Epoch 10/15 | Train Loss: 2.6014 | Val MAE: 2.6009 | Val RMSE: 6.6210
Epoch 11/15 | Train Loss: 2.4638 | Val MAE: 2.3889 | Val RMSE: 6.3103
Epoch 12/15 | Train Loss: 2.3525 | Val MAE: 2.3180 | Val RMSE: 6.1141
Epoch 13/15 | Train Loss: 2.2273 | Val MAE: 2.2525 | Val RMSE: 6.0795
Epoch 14/15 | Train Loss: 2.1616 | Val MAE: 2.2316 | Val RMSE: 5.9139
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch,15
test_loss,2.25732
test_mae,2.25732


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,maxvit_image_only,2.257317,6.100521,2.408862,0.958285,5.607069,4.023537,0.458274,2.006679,0.338514


## MiniLM Experiments

In [35]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

In [36]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.3971 | Val MAE: 8.2117 | Val RMSE: 18.6801
Epoch 2/15 | Train Loss: 6.7976 | Val MAE: 4.6953 | Val RMSE: 11.2413
Epoch 3/15 | Train Loss: 4.7038 | Val MAE: 3.7464 | Val RMSE: 9.0268
Epoch 4/15 | Train Loss: 3.9862 | Val MAE: 3.0370 | Val RMSE: 8.1387
Epoch 5/15 | Train Loss: 3.6504 | Val MAE: 3.2369 | Val RMSE: 8.7668
Epoch 6/15 | Train Loss: 3.3899 | Val MAE: 2.7041 | Val RMSE: 7.3468
Epoch 7/15 | Train Loss: 3.1449 | Val MAE: 2.7271 | Val RMSE: 7.1826
Epoch 8/15 | Train Loss: 2.9368 | Val MAE: 2.6036 | Val RMSE: 6.9008
Epoch 9/15 | Train Loss: 2.7871 | Val MAE: 3.3563 | Val RMSE: 7.9567
Epoch 10/15 | Train Loss: 2.5917 | Val MAE: 2.4271 | Val RMSE: 6.6022
Epoch 11/15 | Train Loss: 2.4507 | Val MAE: 2.3504 | Val RMSE: 6.2337
Epoch 12/15 | Train Loss: 2.3653 | Val MAE: 2.3040 | Val RMSE: 6.3476
Epoch 13/15 | Train Loss: 2.2390 | Val MAE: 2.4108 | Val RMSE: 6.3196
Epoch 14/15 | Train Loss: 2.1794 | Val MAE: 2.3427 | Val RMSE: 6.0964
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▂▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▂▁▁▁▁▁▁
val_rmse,█▄▃▂▃▂▂▁▂▁▁▁▁▁▁
best_val_mae,2.26465
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.138708,5.105526,3.032415,0.948289,4.898622,3.190592,0.506646,2.063298,0.331093,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.935829,"[3.0324154, 0.9482885, 4.8986216, 3.190592, 0...."
0,maxvit_image_only,2.257317,6.100521,2.408862,0.958285,5.607069,4.023537,0.458274,2.006679,0.338514,NaN,NaN,NaN,NaN,NaN
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.341063,5.568154,3.231096,0.961211,5.516939,3.509894,0.470821,2.292354,0.405122,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.111879,"[3.2310963, 0.96121067, 5.516939, 3.5098937, 0..."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.362616,6.162873,2.723720,0.944178,5.802430,4.167202,0.551180,1.963566,0.386038,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.264651,"[2.7237198, 0.9441782, 5.80243, 4.1672025, 0.5..."


## Remote CLIP Experiments

In [37]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_remoteclip_text(
            text_col=text_col,
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6226 | Val MAE: 8.6517 | Val RMSE: 19.7536
Epoch 2/15 | Train Loss: 7.1410 | Val MAE: 4.7202 | Val RMSE: 11.1603
Epoch 3/15 | Train Loss: 4.8616 | Val MAE: 3.3960 | Val RMSE: 8.5221
Epoch 4/15 | Train Loss: 4.0763 | Val MAE: 2.9309 | Val RMSE: 8.0314
Epoch 5/15 | Train Loss: 3.6256 | Val MAE: 2.8111 | Val RMSE: 7.6491
Epoch 6/15 | Train Loss: 3.3679 | Val MAE: 2.8778 | Val RMSE: 7.2439
Epoch 7/15 | Train Loss: 3.1331 | Val MAE: 2.5566 | Val RMSE: 6.7538
Epoch 8/15 | Train Loss: 2.8398 | Val MAE: 2.4404 | Val RMSE: 6.2828
Epoch 9/15 | Train Loss: 2.6968 | Val MAE: 2.4053 | Val RMSE: 6.1490
Epoch 10/15 | Train Loss: 2.5223 | Val MAE: 2.3117 | Val RMSE: 5.8650
Epoch 11/15 | Train Loss: 2.3846 | Val MAE: 2.2291 | Val RMSE: 5.6945
Epoch 12/15 | Train Loss: 2.2518 | Val MAE: 2.0765 | Val RMSE: 5.3979
Epoch 13/15 | Train Loss: 2.1349 | Val MAE: 2.0389 | Val RMSE: 5.2010
Epoch 14/15 | Train Loss: 2.0379 | Val MAE: 1.9567 | Val RMSE: 5.0029


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,1.95239
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6536 | Val MAE: 8.8307 | Val RMSE: 20.0872
Epoch 2/15 | Train Loss: 7.2473 | Val MAE: 4.8058 | Val RMSE: 11.4795
Epoch 3/15 | Train Loss: 4.8241 | Val MAE: 3.5545 | Val RMSE: 8.6308
Epoch 4/15 | Train Loss: 3.9595 | Val MAE: 3.0397 | Val RMSE: 8.0804
Epoch 5/15 | Train Loss: 3.5130 | Val MAE: 2.6695 | Val RMSE: 7.1949
Epoch 6/15 | Train Loss: 3.2432 | Val MAE: 2.5298 | Val RMSE: 6.5508
Epoch 7/15 | Train Loss: 3.0192 | Val MAE: 2.4206 | Val RMSE: 6.3466
Epoch 8/15 | Train Loss: 2.7243 | Val MAE: 2.3547 | Val RMSE: 5.7683
Epoch 9/15 | Train Loss: 2.5376 | Val MAE: 2.2594 | Val RMSE: 5.6658
Epoch 10/15 | Train Loss: 2.3778 | Val MAE: 2.0783 | Val RMSE: 5.2420
Epoch 11/15 | Train Loss: 2.2575 | Val MAE: 2.0504 | Val RMSE: 5.1012
Epoch 12/15 | Train Loss: 2.1546 | Val MAE: 1.9988 | Val RMSE: 4.8676
Epoch 13/15 | Train Loss: 2.0580 | Val MAE: 2.0552 | Val RMSE: 4.9336
Epoch 14/15 | Train Loss: 1.9768 | Val MAE: 1.8999 | Val RMSE: 4.6223


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.89985
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6050 | Val MAE: 8.6567 | Val RMSE: 19.6903
Epoch 2/15 | Train Loss: 7.1320 | Val MAE: 4.6981 | Val RMSE: 11.1303
Epoch 3/15 | Train Loss: 4.8644 | Val MAE: 3.3743 | Val RMSE: 8.5601
Epoch 4/15 | Train Loss: 4.0877 | Val MAE: 3.0308 | Val RMSE: 8.2046
Epoch 5/15 | Train Loss: 3.6856 | Val MAE: 2.9906 | Val RMSE: 8.1574
Epoch 6/15 | Train Loss: 3.5073 | Val MAE: 2.7831 | Val RMSE: 7.4655
Epoch 7/15 | Train Loss: 3.2781 | Val MAE: 2.6722 | Val RMSE: 7.3560
Epoch 8/15 | Train Loss: 3.0148 | Val MAE: 2.7795 | Val RMSE: 7.2221
Epoch 9/15 | Train Loss: 2.8251 | Val MAE: 2.7470 | Val RMSE: 7.1531
Epoch 10/15 | Train Loss: 2.6606 | Val MAE: 2.3858 | Val RMSE: 6.5765
Epoch 11/15 | Train Loss: 2.5004 | Val MAE: 2.6044 | Val RMSE: 6.7242
Epoch 12/15 | Train Loss: 2.4101 | Val MAE: 2.2306 | Val RMSE: 6.1258
Epoch 13/15 | Train Loss: 2.2669 | Val MAE: 2.2353 | Val RMSE: 6.0918
Epoch 14/15 | Train Loss: 2.1639 | Val MAE: 2.1649 | Val RMSE: 5.7762


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▂▂▂▂▂▂▂▁▁▁▁▁▁
val_mae,█▄▂▂▂▂▂▂▂▁▁▁▁▁▁
val_rmse,█▄▂▂▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.16493
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,2.030483,4.915475,2.288666,0.951082,5.102625,3.172665,0.433416,1.912466,0.352458,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.899854,"[2.2886662, 0.951082, 5.102625, 3.1726649, 0.4..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,2.044621,5.217069,2.616254,0.948887,4.741227,3.073761,0.496447,1.994457,0.441311,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.952393,"[2.6162543, 0.94888747, 4.741227, 3.0737615, 0..."
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.138708,5.105526,3.032415,0.948289,4.898622,3.190592,0.506646,2.063298,0.331093,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.935829,"[3.0324154, 0.9482885, 4.8986216, 3.190592, 0...."
0,maxvit_image_only,2.257317,6.100521,2.408862,0.958285,5.607069,4.023537,0.458274,2.006679,0.338514,NaN,NaN,NaN,NaN,NaN
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.341063,5.568154,3.231096,0.961211,5.516939,3.509894,0.470821,2.292354,0.405122,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.111879,"[3.2310963, 0.96121067, 5.516939, 3.5098937, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,2.360599,6.452744,2.466667,0.958069,5.531898,4.445001,0.502652,2.137413,0.482494,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.164927,"[2.4666674, 0.9580694, 5.5318975, 4.445001, 0...."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.362616,6.162873,2.723720,0.944178,5.802430,4.167202,0.551180,1.963566,0.386038,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.264651,"[2.7237198, 0.9441782, 5.80243, 4.1672025, 0.5..."


## DistilBERT Experiments

In [38]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.7339 | Val MAE: 9.0508 | Val RMSE: 20.4017
Epoch 2/15 | Train Loss: 7.3653 | Val MAE: 4.7894 | Val RMSE: 11.3727
Epoch 3/15 | Train Loss: 4.9888 | Val MAE: 3.9296 | Val RMSE: 9.6615
Epoch 4/15 | Train Loss: 4.2248 | Val MAE: 3.1156 | Val RMSE: 8.3922
Epoch 5/15 | Train Loss: 3.8284 | Val MAE: 2.9195 | Val RMSE: 7.6018
Epoch 6/15 | Train Loss: 3.4524 | Val MAE: 2.7346 | Val RMSE: 7.2951
Epoch 7/15 | Train Loss: 3.2462 | Val MAE: 2.6074 | Val RMSE: 6.9997
Epoch 8/15 | Train Loss: 3.0038 | Val MAE: 2.9009 | Val RMSE: 7.1641
Epoch 9/15 | Train Loss: 2.8408 | Val MAE: 2.4392 | Val RMSE: 6.4464
Epoch 10/15 | Train Loss: 2.6378 | Val MAE: 2.5622 | Val RMSE: 6.5835
Epoch 11/15 | Train Loss: 2.5423 | Val MAE: 2.6126 | Val RMSE: 6.8718
Epoch 12/15 | Train Loss: 2.4157 | Val MAE: 2.2615 | Val RMSE: 6.1890
Epoch 13/15 | Train Loss: 2.2881 | Val MAE: 2.2424 | Val RMSE: 6.1276
Epoch 14/15 | Train Loss: 2.2058 | Val MAE: 2.3470 | Val RMSE: 5.9710
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁
best_val_mae,2.1817
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,1.955852,4.728351,2.529707,0.947564,4.564701,2.968892,0.428522,1.883379,0.368201,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.856157,"[2.5297074, 0.94756424, 4.5647006, 2.9688916, ..."
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,2.030483,4.915475,2.288666,0.951082,5.102625,3.172665,0.433416,1.912466,0.352458,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.899854,"[2.2886662, 0.951082, 5.102625, 3.1726649, 0.4..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,2.044621,5.217069,2.616254,0.948887,4.741227,3.073761,0.496447,1.994457,0.441311,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.952393,"[2.6162543, 0.94888747, 4.741227, 3.0737615, 0..."
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.138708,5.105526,3.032415,0.948289,4.898622,3.190592,0.506646,2.063298,0.331093,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.935829,"[3.0324154, 0.9482885, 4.8986216, 3.190592, 0...."
0,maxvit_image_only,2.257317,6.100521,2.408862,0.958285,5.607069,4.023537,0.458274,2.006679,0.338514,NaN,NaN,NaN,NaN,NaN
9,maxvit_vision_gemma3_4b_clean_distilbert-base-...,2.302543,6.175428,2.736033,0.950496,5.585752,4.143747,0.460512,1.879563,0.361696,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.181696,"[2.736033, 0.9504964, 5.5857515, 4.1437473, 0...."
7,maxvit_hybrid_gemma3_4b_clean_distilbert-base-...,2.310054,6.089441,2.613521,0.950653,5.639628,4.209648,0.466056,1.957540,0.333333,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.189174,"[2.6135213, 0.9506528, 5.6396284, 4.2096477, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.341063,5.568154,3.231096,0.961211,5.516939,3.509894,0.470821,2.292354,0.405122,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.111879,"[3.2310963, 0.96121067, 5.516939, 3.5098937, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,2.360599,6.452744,2.466667,0.958069,5.531898,4.445001,0.502652,2.137413,0.482494,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.164927,"[2.4666674, 0.9580694, 5.5318975, 4.445001, 0...."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.362616,6.162873,2.723720,0.944178,5.802430,4.167202,0.551180,1.963566,0.386038,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.264651,"[2.7237198, 0.9441782, 5.80243, 4.1672025, 0.5..."


## MaxVit Experiment Results Summary

In [39]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,1.955852,4.728351,2.529707,0.947564,4.564701,2.968892,0.428522,1.883379,0.368201,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.856157,"[2.5297074, 0.94756424, 4.5647006, 2.9688916, ..."
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,2.030483,4.915475,2.288666,0.951082,5.102625,3.172665,0.433416,1.912466,0.352458,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.899854,"[2.2886662, 0.951082, 5.102625, 3.1726649, 0.4..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,2.044621,5.217069,2.616254,0.948887,4.741227,3.073761,0.496447,1.994457,0.441311,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.952393,"[2.6162543, 0.94888747, 4.741227, 3.0737615, 0..."
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.138708,5.105526,3.032415,0.948289,4.898622,3.190592,0.506646,2.063298,0.331093,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.935829,"[3.0324154, 0.9482885, 4.8986216, 3.190592, 0...."
0,maxvit_image_only,2.257317,6.100521,2.408862,0.958285,5.607069,4.023537,0.458274,2.006679,0.338514,NaN,NaN,NaN,NaN,NaN
9,maxvit_vision_gemma3_4b_clean_distilbert-base-...,2.302543,6.175428,2.736033,0.950496,5.585752,4.143747,0.460512,1.879563,0.361696,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.181696,"[2.736033, 0.9504964, 5.5857515, 4.1437473, 0...."
7,maxvit_hybrid_gemma3_4b_clean_distilbert-base-...,2.310054,6.089441,2.613521,0.950653,5.639628,4.209648,0.466056,1.957540,0.333333,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.189174,"[2.6135213, 0.9506528, 5.6396284, 4.2096477, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.341063,5.568154,3.231096,0.961211,5.516939,3.509894,0.470821,2.292354,0.405122,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.111879,"[3.2310963, 0.96121067, 5.516939, 3.5098937, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,2.360599,6.452744,2.466667,0.958069,5.531898,4.445001,0.502652,2.137413,0.482494,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.164927,"[2.4666674, 0.9580694, 5.5318975, 4.445001, 0...."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.362616,6.162873,2.723720,0.944178,5.802430,4.167202,0.551180,1.963566,0.386038,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.264651,"[2.7237198, 0.9441782, 5.80243, 4.1672025, 0.5..."


### MaxViT Results Summary

Experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

The image-only baseline achieves a test MAE of 2.36, slightly better than SwinV2, indicating stronger visual feature extraction.

Incorporating textual information leads to improvements, with the best performance obtained using text-only captions (MAE: ~1.94–1.99). However, the magnitude of improvement is smaller compared to SwinV2.

Hybrid captions provide limited gains over the baseline, and in some cases only marginal improvement. Vision-based captions again do not improve performance and remain close to the image only baseline.

Class-wise results show modest improvements, particularly for dominant classes such as Grass and Crop, while changes for other classes remain relatively small.

Overall, while textual information still improves performance, the gains are less pronounced compared to SwinV2, suggesting that the stronger MaxViT backbone already captures more informative visual features.

## Text Scale Experiments

The best caption × encoder combination for MaxVit based on
validation MAE at scale 0.7 is selected.

In [40]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for MaxVit (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for MaxVit (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 1.8562
  Test MAE    : 1.9559


## Text Scale Ablation

Using the best configuration selected above, we sweep text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5].

In [41]:
SCALE_WANDB_PROJECT = "DI725-Project-landcover-regression_phase3_text_scale_exp"
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_maxvit_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=SCALE_WANDB_PROJECT,
            save_predictions=True
        )
    else:
        result = run_maxvit_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=SCALE_WANDB_PROJECT,
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.7337 | Val MAE: 9.0225 | Val RMSE: 20.3399
Epoch 2/15 | Train Loss: 7.2862 | Val MAE: 4.6667 | Val RMSE: 11.2123
Epoch 3/15 | Train Loss: 4.8998 | Val MAE: 4.0742 | Val RMSE: 9.4512
Epoch 4/15 | Train Loss: 4.1147 | Val MAE: 2.9689 | Val RMSE: 8.0713
Epoch 5/15 | Train Loss: 3.6442 | Val MAE: 2.9130 | Val RMSE: 7.4471
Epoch 6/15 | Train Loss: 3.3167 | Val MAE: 2.8070 | Val RMSE: 6.9335
Epoch 7/15 | Train Loss: 3.0548 | Val MAE: 2.6011 | Val RMSE: 6.8288
Epoch 8/15 | Train Loss: 2.8435 | Val MAE: 2.4790 | Val RMSE: 6.2300
Epoch 9/15 | Train Loss: 2.6365 | Val MAE: 2.5262 | Val RMSE: 6.0288
Epoch 10/15 | Train Loss: 2.4824 | Val MAE: 2.1905 | Val RMSE: 5.6491
Epoch 11/15 | Train Loss: 2.3563 | Val MAE: 2.1822 | Val RMSE: 5.3827
Epoch 12/15 | Train Loss: 2.2246 | Val MAE: 2.1571 | Val RMSE: 5.4294
Epoch 13/15 | Train Loss: 2.1188 | Val MAE: 2.0030 | Val RMSE: 5.0685
Epoch 14/15 | Train Loss: 2.0517 | Val MAE: 2.0632 | Val RMSE: 5.2278
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.96212
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.7613 | Val MAE: 8.9851 | Val RMSE: 20.3163
Epoch 2/15 | Train Loss: 7.2212 | Val MAE: 4.7643 | Val RMSE: 11.3138
Epoch 3/15 | Train Loss: 4.7832 | Val MAE: 3.4908 | Val RMSE: 8.6770
Epoch 4/15 | Train Loss: 3.9770 | Val MAE: 2.9085 | Val RMSE: 7.4130
Epoch 5/15 | Train Loss: 3.5656 | Val MAE: 2.5819 | Val RMSE: 6.6837
Epoch 6/15 | Train Loss: 3.2230 | Val MAE: 2.5946 | Val RMSE: 6.5873
Epoch 7/15 | Train Loss: 2.9870 | Val MAE: 2.5642 | Val RMSE: 6.4621
Epoch 8/15 | Train Loss: 2.7608 | Val MAE: 2.3086 | Val RMSE: 5.8609
Epoch 9/15 | Train Loss: 2.5541 | Val MAE: 2.1178 | Val RMSE: 5.1927
Epoch 10/15 | Train Loss: 2.4244 | Val MAE: 2.1531 | Val RMSE: 5.2103
Epoch 11/15 | Train Loss: 2.3016 | Val MAE: 2.0085 | Val RMSE: 4.7263
Epoch 12/15 | Train Loss: 2.1783 | Val MAE: 1.9180 | Val RMSE: 4.5404
Epoch 13/15 | Train Loss: 2.1034 | Val MAE: 1.8956 | Val RMSE: 4.6517
Epoch 14/15 | Train Loss: 2.0259 | Val MAE: 1.9555 | Val RMSE: 4.8036
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.77951
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,1.962125,1.998019,5.030600,"[2.4335737, 0.9516337, 4.697273, 3.2179744, 0....",2.433574,0.951634,4.697273,3.217974,0.470828,1.924072,0.290777
1,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,1.842396,1.966422,4.809547,"[2.562236, 0.95320374, 4.549757, 3.0640795, 0....",2.562236,0.953204,4.549757,3.064080,0.399487,1.854805,0.381390
2,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.865246,1.940822,4.738384,"[2.4877408, 0.9598516, 4.4541883, 3.0111096, 0...",2.487741,0.959852,4.454188,3.011110,0.426854,1.810607,0.435400
3,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,1.844170,1.872147,4.641170,"[2.2499268, 0.94989437, 4.332338, 2.8153381, 0...",2.249927,0.949894,4.332338,2.815338,0.438207,1.918867,0.400461
4,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,1.779507,1.917064,4.647670,"[2.350429, 0.9542028, 4.438962, 3.0658119, 0.4...",2.350429,0.954203,4.438962,3.065812,0.478108,1.809613,0.322322


## Text Scale Ablation — Overall MAE Summary

In [42]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.2573

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,1.962125,1.998019,5.030600,0.2593,11.49
1,0.5,1.842396,1.966422,4.809547,0.2909,12.89
2,0.7,1.865246,1.940822,4.738384,0.3165,14.02
3,1.0,1.844170,1.872147,4.641170,0.3852,17.06
4,1.5,1.779507,1.917064,4.647670,0.3403,15.08


## Text Scale Ablation — Class-wise MAE

In [43]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.433574 0.951634 4.697273 3.217974  0.470828 1.924072 0.290777
        0.5 2.562236 0.953204 4.549757 3.064080  0.399487 1.854805 0.381390
        0.7 2.487741 0.959852 4.454188 3.011110  0.426854 1.810607 0.435400
        1.0 2.249927 0.949894 4.332338 2.815338  0.438207 1.918867 0.400461
        1.5 2.350429 0.954203 4.438962 3.065812  0.478108 1.809613 0.322322

Class-wise improvement vs image-only baseline (positive = better)
 text_scale    Tree   Shrub  Grass   Crop  Built-up  Barren   Water
        0.3 -0.0247  0.0067 0.9098 0.8056   -0.0126  0.0826  0.0477
        0.5 -0.1534  0.0051 1.0573 0.9595    0.0588  0.1519 -0.0429
        0.7 -0.0789 -0.0016 1.1529 1.0124    0.0314  0.1961 -0.0969
        1.0  0.1589  0.0084 1.2747 1.2082    0.0201  0.0878 -0.0619
        1.5  0.0584  0.0041 1.1681 0.9577   -0.0198  0.1971  0.0162


## Full Results Summary — All MaxVit Experiments

In [44]:
scale_result_experiments = {r["experiment"] for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"  if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"     if r.get("experiment") in scale_result_experiments
    else  "multimodal_0.7"),
    axis=1
)

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,NaN,NaN,NaN,2.257317,6.100521
1,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.899854,2.030483,4.915475
2,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.952393,2.044621,5.217069
3,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.935829,2.138708,5.105526
4,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.181696,2.302543,6.175428
5,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.189174,2.310054,6.089441
6,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.111879,2.341063,5.568154
7,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.164927,2.360599,6.452744
8,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.264651,2.362616,6.162873
9,scale_sweep,text_qwen3_4b_clean,distilbert-base-uncased,1.0,1.844170,1.872147,4.641170


In [45]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 1.5
  Val MAE         : 1.7795
  Test MAE        : 1.9171
  Test RMSE       : 4.6477


In [46]:
BACKBONE_NAME = "maxvit"
all_results_df.to_csv(
    f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/all_results_{BACKBONE_NAME}.csv",
    index=False
)
print("Saved results to Drive")

Saved results to Drive
